# FATFORMER-XLA: TASK 7.11 — HUẤN LUYỆN ANCHORED RECOVERY v2
## Môi trường: Google Colab GPU (T4 / L4 / A100)

> **Mục tiêu cốt lõi (Cổng G7.2)**:
- Huấn luyện checkpoint `fatformer_anchored_v2_best.pth` với các thông số kế thừa đã được nghiệm thu:
  + **Kho dữ liệu v2**: 14.400 mẫu đa dạng, cách ly 100% rò rỉ (`violations = 0`).
  + **Degradation Pipeline v2**: 40% suy biến mô phỏng mạng xã hội 2024-2025 (WebP, Composite, Double-JPEG).
  + **Hàm mất mát Loss v2**: $0.3\mathcal{L}_{CE\_clean} + 0.3\mathcal{L}_{CE\_deg} + 0.7\mathcal{L}_{KD} + 0.2\mathcal{L}_{cons} + 0.3\mathcal{L}_{feat} + 0.5\mathcal{L}_{guard}$.
  + **Selection Metric v2**: Tối ưu Worst-Group Score trên 4 nhóm với ràng buộc an toàn Real ACC $\ge 90.0\%$.
- **Mục tiêu năng lực G7.2**: Fake@Real90 $\ge 65\%$, AUC $\ge 78\%$, AUC$_{Q30} \ge 60\%$, Macro ACC $\ge 90\%$, Real ACC $\ge 90\%$, ECE $\le 0.08$.

### BƯỚC 1: KẾT NỐI GOOGLE DRIVE & KIỂM TRA PHẦN CỨNG GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Kiem tra thong so GPU
!nvidia-smi

### BƯỚC 2: THIẾT LẬP MÃ NGUỒN TỪ GITHUB (NHÁNH `hoang`)

In [ ]:
import os

%cd /content
if not os.path.exists('/content/fatformer-xla'):
    !git clone -b hoang https://github.com/LeeVHoangtk3/fatformer-xla.git
%cd /content/fatformer-xla
!git checkout hoang
!git pull origin hoang

# Cai dat cac goi phu thuoc
!pip install -q ftfy regex tqdm pyyaml timm einops

### BƯỚC 3: SAO CHÉP CHECKPOINT & GIẢI NÉN KHO DỮ LIỆU v2 VÀO SSD CỤC BỘ
> **Lưu ý tối ưu I/O**: Giải nén tập dữ liệu 7.78 GB từ Google Drive sang ổ đĩa SSD cục bộ `/content/dataset_local` giúp GPU đọc ảnh với tốc độ tối đa, ngăn chặn hiện tượng nghẽn cổ chai Google Drive I/O.

In [ ]:
import os
import shutil

os.makedirs("/content/dataset_local", exist_ok=True)
os.makedirs("/content/fatformer-xla/DATASET/pretrained", exist_ok=True)
os.makedirs("/content/fatformer-xla/DATASET/manifests", exist_ok=True)
os.makedirs("/content/fatformer-xla/DATASET/logs/phase7", exist_ok=True)
os.makedirs("/content/fatformer-xla/DATASET/checkpoints/phase7", exist_ok=True)

# 1. Sao chep mo hinh tien huan luyen
print("[*] Sao chep mo hinh tien huan luyen tu Google Drive...")
!cp /content/drive/MyDrive/Fatformer/pretrained/ViT-L-14.pt /content/fatformer-xla/DATASET/pretrained/
!cp /content/drive/MyDrive/Fatformer/pretrained/fatformer_4class_ckpt.pth /content/fatformer-xla/DATASET/pretrained/

# 2. Sao chep Manifest v2 va bao cao co lap
print("[*] Sao chep Manifest v2...")
!cp /content/drive/MyDrive/Fatformer/phase7/data_manifest_v2.json /content/fatformer-xla/DATASET/manifests/
!cp /content/drive/MyDrive/Fatformer/phase7/data_isolation_report.json /content/fatformer-xla/DATASET/logs/phase7/

# 3. Giai nen kho du lieu v2 vao SSD cuc bo
tar_drive_path = "/content/drive/MyDrive/Fatformer/phase7/train_crossdomain_v2.tar"
if os.path.exists(tar_drive_path):
    print(f"[*] Dang giai nen kho du lieu {tar_drive_path} vao /content/dataset_local (7.78 GB)... chu y doi 2-3 phut...")
    !tar -xf /content/drive/MyDrive/Fatformer/phase7/train_crossdomain_v2.tar -C /content/dataset_local/
    print("[✓] Giai nen thanh cong!")
else:
    print(f"[!] Canh bao: Khong tim thay {tar_drive_path}. Vui long kiem tra lai duong dan tren Drive.")

### BƯỚC 4: CHẠY PRE-FLIGHT CHECKLIST (KIỂM CHỨNG 8 CHỐT CHẶN AN TOÀN)

In [ ]:
# Chay script kiem dinh tu dong 8 chot chan bat buoc truoc khi train
!python tools/preflight_phase7_train.py

### BƯỚC 5: BENCHMARK 100 STEPS (ĐO TỐC ĐỘ, VRAM & ƯỚC LƯỢNG THỜI GIAN)

In [ ]:
# Chay benchmark 100 steps dau tien de xac dinh toc do ms/step va an toan VRAM
!python -m src.training.train_crossdomain --config configs/train_phase7_anchored_v2.yaml --max_steps 100

### BƯỚC 6: TIẾN HÀNH HUẤN LUYỆN CHÍNH THỨC 5 EPOCHS (TASK 7.11)
- Checkpoint tốt nhất sẽ tự động được chọn theo **Worst-Group Score** có bảo vệ Real ACC $\ge 90.0\%$.
- Checkpoint và nhật ký `val_groups.csv` tự động đồng bộ sang Google Drive `/content/drive/MyDrive/Fatformer/checkpoint/phase7/` sau mỗi epoch.

In [ ]:
!python -m src.training.train_crossdomain --config configs/train_phase7_anchored_v2.yaml

### BƯỚC 7: TRÍCH XUẤT BẢNG NHẬT KÝ & VẼ BIỂU ĐỒ HỘI TỤ (DoD)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

csv_path = "DATASET/checkpoints/phase7/val_groups.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=" * 80)
    print("NHẬT KÝ ĐÁNH GIÁ 4 NHÓM RECALL QUA TỪNG EPOCH (val_groups.csv):")
    print("=" * 80)
    print(df.to_string(index=False))
    
    # Ve bieu do hoi tu
    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['clean_r'] * 100, marker='o', label='Clean Real Recall (%)')
    plt.plot(df['epoch'], df['clean_f'] * 100, marker='s', label='Clean Fake Recall (%)')
    plt.plot(df['epoch'], df['q30_r'] * 100, marker='^', label='Q30 Real Recall (%)')
    plt.plot(df['epoch'], df['q30_f'] * 100, marker='d', label='Q30 Fake Recall (%)')
    plt.plot(df['epoch'], df['worst'] * 100, marker='*', linewidth=2.5, color='red', label='Worst-Group Score (%)')
    plt.axhline(y=90.0, color='gray', linestyle='--', label='Nguong an toan Real (90%)')
    
    plt.title("FatFormer-XLA Phase 7: Tien trinh hoi tu 4 nhom (Task 7.11)", fontsize=14)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Recall / Score (%)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='lower right')
    
    out_chart = "DATASET/logs/phase7/m7_convergence.png"
    plt.savefig(out_chart, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"[✓] Da xuat bieu do hoi tu thanh cong tai: {out_chart}")
else:
    print(f"[!] Khong tim thay file {csv_path}")